# cdslib — generic OIS generator demo

Play with the OIS generators: change the rate index, tenor code and trade
date and re-run cells. Everything is driven by the JSON data under
`src/cdslib/data/` (`rate_indices.json`, `holidays.json`).

> Holidays are seeded for **2024–2026** only; dates outside that range fall
> back to a weekend-only calendar.

In [4]:
from datetime import date

from cdslib import (
    CalendarRegistry,
    OISGenerator,
    RateIndexRegistry,
)

TRADE_DATE = date(2026, 5, 4)
gen = OISGenerator.load_default()

## 1. Available rate indices and their conventions
Loaded straight from `rate_indices.json`.

In [5]:
reg = RateIndexRegistry.load_default()
for name in reg.indices():
    c = reg.get(name)
    print(
        f"{name:8s} ccy={c.currency} dc={c.day_count!s:8s} "
        f"spot_lag={c.spot_lag} pay_lag={c.payment_lag} "
        f"freq={c.fixed_frequency} bdc={c.business_day_convention}"
    )

RUONIA   ccy=RUB dc=ACT/365F spot_lag=1 pay_lag=1 freq=1Y bdc=MODIFIED_FOLLOWING
SOFR     ccy=USD dc=ACT/360  spot_lag=2 pay_lag=2 freq=1Y bdc=MODIFIED_FOLLOWING


## 2. Generate a single OIS and inspect its schedule
`show()` prints the fixed-leg accrual periods, payment dates and year
fractions. The floating leg mirrors the same schedule.

In [7]:
def show(swap):
    print(f"{swap.rate_index} {swap.tenor}  ccy={swap.currency}  dc={swap.day_count}")
    print(
        f"trade={swap.trade_date}  spot={swap.spot_date}  maturity={swap.maturity_date}"
    )
    print(
        f"{'#':>2}  {'accrual_start':<12} {'accrual_end':<12} {'payment':<12} {'year_frac':>9}"
    )
    for i, p in enumerate(swap.fixed_periods, 1):
        print(
            f"{i:>2}  {p.accrual_start!s:<12} {p.accrual_end!s:<12} "
            f"{p.payment_date!s:<12} {p.year_fraction:>9.5f}"
        )
    total = sum(p.year_fraction for p in swap.fixed_periods)
    print(f"sum(year_fraction) = {total:.5f}")


show(gen.generate("RUONIA", "1w", TRADE_DATE))

RUONIA 1W  ccy=RUB  dc=ACT/365F
trade=2026-05-04  spot=2026-05-05  maturity=2026-05-12
 #  accrual_start accrual_end  payment      year_frac
 1  2026-05-05   2026-05-12   2026-05-13     0.01918
sum(year_fraction) = 0.01918


## 3. Sweep across tenor codes
Short tenors (≤ 1Y) collapse to a single period; longer ones pay annually.

In [8]:
for code in ["1w", "1m", "3m", "6m", "9m", "1y", "2y", "3y", "5y", "10y"]:
    s = gen.generate("RUONIA", code, TRADE_DATE)
    print(
        f"{code:>4}: periods={len(s.fixed_periods):>2}  spot={s.spot_date}  maturity={s.maturity_date}"
    )

  1w: periods= 1  spot=2026-05-05  maturity=2026-05-12
  1m: periods= 1  spot=2026-05-05  maturity=2026-06-05
  3m: periods= 1  spot=2026-05-05  maturity=2026-08-05
  6m: periods= 1  spot=2026-05-05  maturity=2026-11-05
  9m: periods= 1  spot=2026-05-05  maturity=2027-02-05
  1y: periods= 1  spot=2026-05-05  maturity=2027-05-05
  2y: periods= 2  spot=2026-05-05  maturity=2028-05-05
  3y: periods= 3  spot=2026-05-05  maturity=2029-05-07
  5y: periods= 5  spot=2026-05-05  maturity=2031-05-05
 10y: periods=10  spot=2026-05-05  maturity=2036-05-05


## 4. Holiday & spot-lag behaviour
RUONIA spot lag is 1 business day, so trading just before a RUB holiday
hops the effective date over it.

In [9]:
cal = CalendarRegistry.load_default().get("RUB")
print(
    "2024-06-12 (Russia Day) is a business day?", cal.is_business_day(date(2024, 6, 12))
)

for trade in [date(2024, 6, 10), date(2024, 6, 11)]:
    s = gen.generate("RUONIA", "1y", trade)
    print(f"trade={trade} -> spot={s.spot_date}  maturity={s.maturity_date}")

2024-06-12 (Russia Day) is a business day? False
trade=2024-06-10 -> spot=2024-06-11  maturity=2025-06-11
trade=2024-06-11 -> spot=2024-06-13  maturity=2025-06-16


## 5. Same code, different market (SOFR / USD)
Nothing is hard-coded to RUONIA — the conventions come from the registry.

In [10]:
show(gen.generate("SOFR", "2y", TRADE_DATE))

SOFR 2Y  ccy=USD  dc=ACT/360
trade=2026-05-04  spot=2026-05-06  maturity=2028-05-08
 #  accrual_start accrual_end  payment      year_frac
 1  2026-05-06   2027-05-06   2027-05-10     1.01389
 2  2027-05-06   2028-05-08   2028-05-10     1.02222
sum(year_fraction) = 2.03611


## 6. Playground
Tweak these and re-run.

In [11]:
RATE = "RUONIA"
TENOR = "3y"
TRADE = date(2024, 6, 3)

show(gen.generate(RATE, TENOR, TRADE))

RUONIA 3Y  ccy=RUB  dc=ACT/365F
trade=2024-06-03  spot=2024-06-04  maturity=2027-06-04
 #  accrual_start accrual_end  payment      year_frac
 1  2024-06-04   2025-06-04   2025-06-05     1.00000
 2  2025-06-04   2026-06-04   2026-06-05     1.00000
 3  2026-06-04   2027-06-04   2027-06-07     1.00000
sum(year_fraction) = 3.00000
